# Stage 8. Backbone Experiment

Notebook ini membandingkan tiga backbone deep embedding (ResNet18-CSA baseline, DINOv2-S self-supervised ViT, SigLIP-B vision-language ViT) pada situs conjunctiva, memakai threshold valid_pixel_mask baru (high=220) berdasarkan analisis empiris notebooks/analysis_valid_pixel.ipynb. Model produksi conjunctiva saat ini adalah Full Fusion ResNet18-CSA tuned (MAE 1.554, accuracy 0.733, AUC ~0.826). Conjunctiva memiliki dua dataset (CP-AnemiC dan Eyes-Defy), sehingga Optuna objective menyertakan komponen worst-case accuracy antar dataset agar solusi tidak collapse di satu populasi.

## Environment Setup

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent.parent))

import json

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import cohen_kappa_score

from configs import paths
from src.common import features, manifest as manifest_utils, train
from src.sites.conjunctiva import data

output_dir = paths.outputs_dir("conjunctiva")
artifact_dir = paths.artifacts_dir("conjunctiva")
manifest = manifest_utils.assign_kfold(data.build_manifest(save=False), n_splits=5, seed=42)

PIXEL_HIGH = 220

print("sampel", len(manifest))
print("device", "cuda" if torch.cuda.is_available() else "cpu")
print("pixel_high", PIXEL_HIGH)

Eyes-Defy Italy melewati 2 folder tanpa metadata atau file lengkap: [93, 95]
Eyes-Defy India melewati 1 folder tanpa metadata atau file lengkap: [7]
sampel 925
device cuda
pixel_high 220


## Re-extract Handcrafted Features (Threshold Baru)

Fitur handcrafted diekstrak ulang dengan pixel_high=220. Pada conjunctiva, penurunan threshold dari 240 ke 220 membuang sekitar 8.6% pixel valid tambahan yang berada di rentang near-glare.

In [2]:
handcrafted = features.extract_handcrafted_features(manifest, pixel_high=PIXEL_HIGH)
handcrafted.to_csv(output_dir / "handcrafted_features_t220.csv", index=False)
print("handcrafted features", handcrafted.shape)

handcrafted features (925, 28)


## Extract Deep Embeddings per Backbone

In [3]:
backbone_resnet = features.EmbeddingBackbone(backbone_name="resnet18", seed=42)
embeddings_resnet, uids_resnet = features.extract_deep_embeddings(
    manifest, model=backbone_resnet, size=backbone_resnet.input_size, pixel_high=PIXEL_HIGH,
)
np.save(output_dir / "deep_embeddings_resnet18_t220.npy", embeddings_resnet)
pd.DataFrame({"uid": uids_resnet}).to_csv(output_dir / "deep_embeddings_resnet18_t220_uids.csv", index=False)
print("resnet18", embeddings_resnet.shape)

resnet18 (925, 256)


In [4]:
backbone_dinov2 = features.EmbeddingBackbone(backbone_name="dinov2_small", seed=42)
embeddings_dinov2, uids_dinov2 = features.extract_deep_embeddings(
    manifest, model=backbone_dinov2, size=backbone_dinov2.input_size, pixel_high=PIXEL_HIGH,
)
np.save(output_dir / "deep_embeddings_dinov2s_t220.npy", embeddings_dinov2)
pd.DataFrame({"uid": uids_dinov2}).to_csv(output_dir / "deep_embeddings_dinov2s_t220_uids.csv", index=False)
print("dinov2_small", embeddings_dinov2.shape)

dinov2_small (925, 256)


In [5]:
backbone_siglip = features.EmbeddingBackbone(backbone_name="siglip_base_patch16_224", seed=42)
embeddings_siglip, uids_siglip = features.extract_deep_embeddings(
    manifest, model=backbone_siglip, size=backbone_siglip.input_size, pixel_high=PIXEL_HIGH,
)
np.save(output_dir / "deep_embeddings_siglip_t220.npy", embeddings_siglip)
pd.DataFrame({"uid": uids_siglip}).to_csv(output_dir / "deep_embeddings_siglip_t220_uids.csv", index=False)
print("siglip_base", embeddings_siglip.shape)

siglip_base (925, 256)


## Backbone Comparison: Path B and Full Fusion

Setiap backbone diuji dengan dua konfigurasi (Path B deep only, Full Fusion) memakai run_kfold dengan protokol identik. Performa per dataset (CP-AnemiC vs Eyes-Defy) juga dilaporkan karena collapse pada satu populasi adalah risiko utama yang teridentifikasi pada Stage 5.

In [6]:
def _severity_kappa(oof):
    severity_valid = oof["severity_true"] >= 0
    if not severity_valid.any():
        return float("nan")
    return cohen_kappa_score(oof.loc[severity_valid, "severity_true"], oof.loc[severity_valid, "severity_pred"])


backbone_configs = {
    "resnet18": (embeddings_resnet, uids_resnet),
    "dinov2_small": (embeddings_dinov2, uids_dinov2),
    "siglip_base": (embeddings_siglip, uids_siglip),
}

summary_rows = []
all_results = {}

for config_type, use_hc in [("path_b_deep", False), ("full_fusion", True)]:
    print(f"\n--- {config_type} ---")
    for backbone_name, (embeddings, uids) in backbone_configs.items():
        result = train.run_kfold(
            manifest, handcrafted, embeddings, uids,
            n_splits=5, epochs=60, use_handcrafted=use_hc, use_deep=True,
        )
        key = f"{backbone_name}_{config_type}"
        all_results[key] = result
        fm = result["fold_metrics"]
        breakdown = train.evaluate_by_dataset(result["oof"], manifest)
        worst_acc = breakdown["accuracy"].min()
        summary_rows.append({
            "backbone": backbone_name,
            "config": config_type,
            "mae": fm["mae"].mean(),
            "accuracy": fm["accuracy"].mean(),
            "auc": fm["auc"].mean(),
            "f1": fm["f1"].mean(),
            "worst_dataset_accuracy": worst_acc,
            "severity_kappa": _severity_kappa(result["oof"]),
        })
        print(
            f"  {backbone_name:20s}",
            f"MAE {fm['mae'].mean():.3f}",
            f"acc {fm['accuracy'].mean():.3f}",
            f"AUC {fm['auc'].mean():.3f}",
            f"worst_acc {worst_acc:.3f}",
        )


--- path_b_deep ---


  resnet18             MAE 1.687 acc 0.641 AUC 0.607 worst_acc 0.610


  dinov2_small         MAE 1.576 acc 0.759 AUC 0.828 worst_acc 0.749


  siglip_base          MAE 1.625 acc 0.669 AUC 0.674 worst_acc 0.646

--- full_fusion ---


  resnet18             MAE 1.613 acc 0.668 AUC 0.711 worst_acc 0.642


  dinov2_small         MAE 1.530 acc 0.784 AUC 0.840 worst_acc 0.777


  siglip_base          MAE 1.592 acc 0.684 AUC 0.726 worst_acc 0.676


## Consolidated Comparison

In [7]:
summary_table = pd.DataFrame(summary_rows)
summary_table.to_csv(output_dir / "backbone_experiment_comparison.csv", index=False)
print(summary_table.sort_values("mae").round(4).to_string(index=False))

    backbone      config    mae  accuracy    auc     f1  worst_dataset_accuracy  severity_kappa
dinov2_small full_fusion 1.5303    0.7838 0.8399 0.7985                  0.7767          0.3071
dinov2_small path_b_deep 1.5761    0.7589 0.8282 0.7818                  0.7488          0.2777
 siglip_base full_fusion 1.5920    0.6844 0.7262 0.7374                  0.6761          0.0122
    resnet18 full_fusion 1.6133    0.6681 0.7105 0.7171                  0.6423          0.0124
 siglip_base path_b_deep 1.6251    0.6692 0.6738 0.7486                  0.6465          0.0129
    resnet18 path_b_deep 1.6869    0.6411 0.6067 0.7415                  0.6099          0.0065


## Hyperparameter Search on Best Backbone

Backbone dengan MAE terendah dan worst-case accuracy tertinggi dioptimasi lewat Optuna. Objective menggabungkan MAE ternormalisasi dengan worst-case accuracy antar dataset, identik dengan pencarian pada Stage 5.

In [8]:
import optuna

CLINICAL_MAE_THRESHOLD = 3.6

best_backbone_row = summary_table.sort_values("mae").iloc[0]
best_backbone_name = best_backbone_row["backbone"]
best_config_type = best_backbone_row["config"]
best_embeddings, best_uids = backbone_configs[best_backbone_name]
use_handcrafted_best = best_config_type == "full_fusion"

print(f"optimasi backbone: {best_backbone_name}, config: {best_config_type}")


def objective(trial):
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)
    epochs = trial.suggest_int("epochs", 30, 120)
    weight_classification = trial.suggest_float("weight_classification", 0.5, 3.0)
    weight_severity = trial.suggest_float("weight_severity", 0.1, 1.0)
    focal_gamma = trial.suggest_float("focal_gamma", 0.5, 3.0)
    dropout = trial.suggest_float("dropout", 0.1, 0.5)
    trunk_dim = trial.suggest_categorical("trunk_dim", [64, 128, 256])
    attention_dim = trial.suggest_categorical("attention_dim", [32, 64, 128])
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])

    result = train.run_kfold(
        manifest, handcrafted, best_embeddings, best_uids,
        n_splits=5, epochs=epochs, batch_size=batch_size, learning_rate=learning_rate,
        loss_weights=(1.0, weight_classification, weight_severity),
        trunk_dim=trunk_dim, attention_dim=attention_dim, dropout=dropout, focal_gamma=focal_gamma,
        use_handcrafted=use_handcrafted_best, use_deep=True,
    )
    mae_mean = float(result["fold_metrics"]["mae"].mean())
    breakdown = train.evaluate_by_dataset(result["oof"], manifest)
    worst_case_accuracy = float(breakdown["accuracy"].min())
    score = mae_mean / CLINICAL_MAE_THRESHOLD + (1.0 - worst_case_accuracy)

    trial.set_user_attr("mae_mean", mae_mean)
    trial.set_user_attr("worst_case_accuracy", worst_case_accuracy)
    trial.set_user_attr("accuracy_mean", float(result["fold_metrics"]["accuracy"].mean()))
    return score


study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=40)

print("best score", round(study.best_value, 4))
print("best MAE", round(study.best_trial.user_attrs["mae_mean"], 4))
print("best accuracy", round(study.best_trial.user_attrs["accuracy_mean"], 4))
print("best worst_case_accuracy", round(study.best_trial.user_attrs["worst_case_accuracy"], 4))
print("best params", study.best_params)

[I 2026-09-30 14:07:34,495] A new study created in memory with name: no-name-50fd2b12-7396-48de-a8f7-9ecc830f5d0d


optimasi backbone: dinov2_small, config: full_fusion


[I 2026-09-30 14:07:38,463] Trial 0 finished with value: 0.7171625754083266 and parameters: {'learning_rate': 0.00019512206624613216, 'epochs': 34, 'weight_classification': 1.353100557708352, 'weight_severity': 0.810044089360868, 'focal_gamma': 2.708004729075646, 'dropout': 0.39054996318099955, 'trunk_dim': 128, 'attention_dim': 32, 'batch_size': 64}. Best is trial 0 with value: 0.7171625754083266.


[I 2026-09-30 14:07:59,668] Trial 1 finished with value: 0.6876861818385063 and parameters: {'learning_rate': 0.005536884454355338, 'epochs': 87, 'weight_classification': 0.7603416702147954, 'weight_severity': 0.5233607557476369, 'focal_gamma': 1.6068137075778692, 'dropout': 0.16950214602153185, 'trunk_dim': 256, 'attention_dim': 32, 'batch_size': 32}. Best is trial 1 with value: 0.6876861818385063.


[I 2026-09-30 14:08:11,924] Trial 2 finished with value: 3.6327538260458407 and parameters: {'learning_rate': 0.004939798263437276, 'epochs': 50, 'weight_classification': 2.5636144889077688, 'weight_severity': 0.6907877604797587, 'focal_gamma': 0.6043480373271279, 'dropout': 0.2758365407918291, 'trunk_dim': 64, 'attention_dim': 32, 'batch_size': 32}. Best is trial 1 with value: 0.6876861818385063.


...

[I 2026-09-30 14:14:09,981] Trial 36 finished with value: 0.6679172594725931 and parameters: {'learning_rate': 0.0007023828792605293, 'epochs': 92, 'weight_classification': 0.7033152814483857, 'weight_severity': 0.5633632195375411, 'focal_gamma': 1.7453806696333027, 'dropout': 0.39124123570287145, 'trunk_dim': 256, 'attention_dim': 128, 'batch_size': 64}. Best is trial 26 with value: 0.6340400704425744.


[I 2026-09-30 14:14:22,687] Trial 37 finished with value: 0.7163558996339359 and parameters: {'learning_rate': 0.00017276538496887155, 'epochs': 113, 'weight_classification': 1.181560498295188, 'weight_severity': 0.6724800918234575, 'focal_gamma': 1.5874747831280283, 'dropout': 0.34243088497401336, 'trunk_dim': 256, 'attention_dim': 32, 'batch_size': 64}. Best is trial 26 with value: 0.6340400704425744.


[I 2026-09-30 14:14:32,886] Trial 38 finished with value: 3.6327538260458407 and parameters: {'learning_rate': 0.0007962979962614031, 'epochs': 89, 'weight_classification': 1.3937543474650853, 'weight_severity': 0.5817501059326926, 'focal_gamma': 0.92606452073355, 'dropout': 0.45139808532287423, 'trunk_dim': 256, 'attention_dim': 128, 'batch_size': 64}. Best is trial 26 with value: 0.6340400704425744.


[I 2026-09-30 14:14:55,582] Trial 39 finished with value: 0.6988345568494279 and parameters: {'learning_rate': 0.003921584828678849, 'epochs': 97, 'weight_classification': 0.9723300712912845, 'weight_severity': 0.8383403379336083, 'focal_gamma': 2.0291306880780375, 'dropout': 0.48574521796116893, 'trunk_dim': 64, 'attention_dim': 128, 'batch_size': 32}. Best is trial 26 with value: 0.6340400704425744.


best score 0.634
best MAE 1.5291
best accuracy 0.7989
best worst_case_accuracy 0.7907
best params {'learning_rate': 0.0009341713907377068, 'epochs': 107, 'weight_classification': 0.9242839022923395, 'weight_severity': 0.7344441015157857, 'focal_gamma': 1.412368228312332, 'dropout': 0.39737068444650486, 'trunk_dim': 256, 'attention_dim': 128, 'batch_size': 64}


## Retrain with Best Hyperparameters

In [9]:
best_params = study.best_params
tuned_result = train.run_kfold(
    manifest, handcrafted, best_embeddings, best_uids,
    n_splits=5,
    epochs=best_params["epochs"],
    batch_size=best_params["batch_size"],
    learning_rate=best_params["learning_rate"],
    loss_weights=(1.0, best_params["weight_classification"], best_params["weight_severity"]),
    trunk_dim=best_params["trunk_dim"],
    attention_dim=best_params["attention_dim"],
    dropout=best_params["dropout"],
    focal_gamma=best_params["focal_gamma"],
    use_handcrafted=use_handcrafted_best, use_deep=True,
)

tuned_fm = tuned_result["fold_metrics"]
breakdown = train.evaluate_by_dataset(tuned_result["oof"], manifest)
print("tuned MAE", round(tuned_fm["mae"].mean(), 4))
print("tuned accuracy", round(tuned_fm["accuracy"].mean(), 4))
print("tuned AUC", round(tuned_fm["auc"].mean(), 4))
print("tuned F1", round(tuned_fm["f1"].mean(), 4))
print("tuned severity_kappa", round(_severity_kappa(tuned_result["oof"]), 4))
print("\nper dataset:")
print(breakdown.round(4).to_string(index=False))

tuned MAE 1.5765
tuned accuracy 0.7968
tuned AUC 0.8515
tuned F1 0.8204
tuned severity_kappa 0.2972

per dataset:
  dataset   n    mae  accuracy  severity_accuracy
cp_anemic 710 1.6340    0.7944             0.5239
eyes_defy 215 1.3869    0.8047                NaN


## Save Checkpoints and Results

In [10]:
tuned_result["oof"].to_csv(output_dir / f"multitask_oof_{best_backbone_name}_{best_config_type}_tuned.csv", index=False)

for fold_index, fold_model in enumerate(tuned_result["models"]):
    checkpoint_path = artifact_dir / f"multitask_{best_backbone_name}_{best_config_type}_tuned_fold{fold_index}.pt"
    torch.save(fold_model.state_dict(), checkpoint_path)

with open(output_dir / "backbone_experiment_best.json", "w") as handle:
    json.dump({
        "backbone": best_backbone_name,
        "config": best_config_type,
        "use_handcrafted": use_handcrafted_best,
        "pixel_high": PIXEL_HIGH,
        "best_params": best_params,
        "mae": study.best_trial.user_attrs["mae_mean"],
        "accuracy": study.best_trial.user_attrs["accuracy_mean"],
        "worst_case_accuracy": study.best_trial.user_attrs["worst_case_accuracy"],
    }, handle, indent=2)

print(f"checkpoint disimpan untuk {best_backbone_name} {best_config_type}")

checkpoint disimpan untuk dinov2_small full_fusion
